# Set up week 4's tables (week 4, day 1)

For the trainer. Run this notebook once in the class workspace before Monday, from your Git folder of the course: `week-04/setup/day-01-setup.ipynb`. Week 3's setup must have run first: it made the catalog `course` and the schema `returns`, and gave the class read access to the schema. The numbered steps, and what to check, are in `day-01-setup.md` beside it.

It makes five tables in `course.returns`, which every day of the week reads: `model_returns` (1,000 returns as filed), `model_features` (the same 1,000 returns, with the features built from them), and the three time-based sets `model_train` (2020 and 2021, 500 returns), `model_valid` (2022, 250) and `model_test` (2023, 250). The rows are in `day-01-course-data.py` beside this notebook. The course made up every row: no name, number or amount belongs to a real taxpayer. Running it again is safe: it makes each table again from the same rows.

Each code cell has a number on its first line. Run cells 1 to 4 in order.

In [ ]:
# 1. Settings. CLASS_GROUP holds the learners. The tables go in course.returns.
CLASS_GROUP = "training-students"  # the class workspace's group of learners

import os
try:
    _path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _folder = os.path.dirname(_path if _path.startswith("/Workspace") else "/Workspace" + _path)
except NameError:  # outside Databricks: the folder the notebook runs in
    _folder = os.getcwd()
course = {"__name__": "course_data"}
exec(open(os.path.join(_folder, "day-01-course-data.py"), encoding="utf-8").read(), course)
print(len(course["returns"]()), "returns to write to course.returns, and", CLASS_GROUP, "gets read access in cell 4.")

In [ ]:
# 2. The five tables, with their comments and rows, and the tag role = label on exam_changed_tax.
for statement in course["table_statements"]():
    spark.sql(statement)
print("Made course.returns.model_returns, model_features, model_train, model_valid and model_test.")

In [ ]:
# 3. Read back what the learners will see. Expected, each as yes, no and empty labels:
#    model_features 1000 rows (160, 640, 200); model_train 500 (80, 320, 100);
#    model_valid 250 (40, 160, 50); model_test 250 (40, 160, 50); deduction_amt empty in 63 rows;
#    case 123 is F-40409-2022, in model_valid, label yes; channel A 648 rows and B 352.
facts = course["setup_facts"]()
for table in ["model_features", "model_train", "model_valid", "model_test"]:
    counts = {r["exam_changed_tax"]: r["n"] for r in spark.sql(
        f"SELECT exam_changed_tax, COUNT(*) AS n FROM course.returns.{table} GROUP BY exam_changed_tax").collect()}
    print(f"{table}: {sum(counts.values())} rows, yes {counts.get('yes', 0)}, no {counts.get('no', 0)}, "
          f"empty {counts.get(None, 0)}; expected {facts[table]}")
empty = spark.sql("SELECT COUNT(*) AS n FROM course.returns.model_features WHERE deduction_amt IS NULL").first()["n"]
print("deduction_amt empty:", empty, "expected", facts["deduction_amt empty"])
case = spark.sql("SELECT return_id, exam_changed_tax FROM course.returns.model_valid WHERE case_id = 123").collect()
print("case 123 in model_valid:", [(r["return_id"], r["exam_changed_tax"]) for r in case], "expected", facts["case 123"])
print("channels:", {r["made_up_channel"]: r["n"] for r in spark.sql(
    "SELECT made_up_channel, COUNT(*) AS n FROM course.returns.model_features GROUP BY made_up_channel").collect()},
      "expected", facts["channels"])

In [ ]:
# 4. Read access for the class. Week 3's setup granted these on the schema, so each new table is
#    readable already; running them again changes nothing.
for statement in [
    f"GRANT USE CATALOG ON CATALOG course TO `{CLASS_GROUP}`",
    f"GRANT USE SCHEMA ON SCHEMA course.returns TO `{CLASS_GROUP}`",
    f"GRANT SELECT ON SCHEMA course.returns TO `{CLASS_GROUP}`",
]:
    spark.sql(statement)
print("Granted USE CATALOG, USE SCHEMA and SELECT on course.returns to", CLASS_GROUP)